# Matrix Reconstruction (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [82]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Step 1: Load the desired dataset (test.pt / all.pt)
Load correlation matrices.

In [83]:
FILE_NAME = 'data_00_20'
WINDOW_SIZE = 724
STRIDE = 10
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_SIZE}_s{STRIDE}'
RUN = 'AE_288dim_norm0002'
RESULTS_JSON_PATH = f'models/{DATASET_NAME}/AE/{RUN}/run_results.json'


if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / FILE_NAME / 'dataset' / DATASET_NAME

dataset_dir = base_dir

DATASET_ORDER = ['train']
DATASET_FILES = {}
for name in DATASET_ORDER:
    candidate = dataset_dir / f'{name}.pt'
    if candidate.exists():
        DATASET_FILES[name] = candidate

if not DATASET_FILES:
    raise FileNotFoundError(
        f'No dataset .pt files found in: {dataset_dir.absolute()}'
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Available datasets: {", ".join(DATASET_FILES.keys())}')

Environment: Local PC
Dataset selected: data_00_20_w724_s10
Available datasets: train


In [84]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        indices = payload.get('indices', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        indices = None
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), indices, meta


SAMPLE_DATASET = next(iter(DATASET_FILES))
sample_corr, sample_indices, sample_meta = load_corr_payload(DATASET_FILES[SAMPLE_DATASET])

print(f'Sample dataset: {SAMPLE_DATASET}')
print(f'Correlation tensor shape: {sample_corr.shape}')
if sample_indices is not None:
    print(f'Indices: {sample_indices[:10]}')
else:
    print('Indices: not found in payload')

Sample dataset: train
Correlation tensor shape: torch.Size([288, 362, 362])
Indices: [203, 369, 213, 185, 124, 96, 245, 171, 256, 179]


## Step 2: Prepare Matrices (Full Dataset)
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.

In [85]:
def prepare_inputs(corr_tensor: torch.Tensor):
    all_np = corr_tensor.numpy().astype(np.float32)
    n_matrices, n_assets, _ = all_np.shape

    n_features = n_assets * (n_assets - 1) // 2
    tril_idx = np.tril_indices(n_assets, k=-1)
    extracted_np = all_np[:, tril_idx[0], tril_idx[1]]
    x_all = torch.from_numpy(extracted_np)

    return all_np, x_all, n_assets, n_features, tril_idx


sample_all_np, sample_x_all, N_ASSETS, N_FEATURES, SAMPLE_TRIL_IDX = prepare_inputs(sample_corr)

print(f"{'='*40}")
print(f"Number of matrices   : {sample_all_np.shape[0]}")
print(f"Original matrix shape: ({N_ASSETS}, {N_ASSETS})")
print(f"Flattened input size : {N_FEATURES}")
print(f"Sample tensor shape  : {tuple(sample_x_all.shape)}")
print(f"{'='*40}")

Number of matrices   : 288
Original matrix shape: (362, 362)
Flattened input size : 65341
Sample tensor shape  : (288, 65341)


## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [86]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=None, dropout_prob: float = 0.02):
        super().__init__()

        # Se hidden_dims è None, creiamo un Linear AE semplice
        if hidden_dims is None:
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, latent_dim, bias=False),
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, input_dim, bias=False),
            )
        else:
            # Caso Deep Autoencoder con Dropout e attivazioni
            dimensions = [input_dim, *hidden_dims, latent_dim]

            # --- ENCODER ---
            encoder_layers = []
            for i in range(len(dimensions) - 1):
                encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
                if i < len(dimensions) - 2:
                    encoder_layers.append(nn.LeakyReLU(0.01))
                    encoder_layers.append(nn.Dropout(dropout_prob))
            self.encoder = nn.Sequential(*encoder_layers)

            # --- DECODER ---
            decoder_dims = dimensions[::-1]
            decoder_layers = []
            for i in range(len(decoder_dims) - 1):
                decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
                if i < len(decoder_dims) - 2:
                    decoder_layers.append(nn.LeakyReLU(0.01))
                #else:
                    #decoder_layers.append(nn.Tanh())
                    
            self.decoder = nn.Sequential(*decoder_layers)

    def architecture_signature(self):
        return [
            [int(layer.in_features), int(layer.out_features)]
            for layer in self.encoder
            if isinstance(layer, nn.Linear)
        ]

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [87]:
def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p


results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

OUTPUT_DIR_NAME = 'analysis_outputs'
analysis_dir = results_path.parent / OUTPUT_DIR_NAME
analysis_dir.mkdir(parents=True, exist_ok=True)

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

model_cfg = results.get('model', {})
latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
model_type_raw = str(model_cfg.get('model_type', '')).strip().lower()
if model_type_raw in {'linear', 'linearae', 'linear_ae', 'linear-ae'}:
    model_type = 'linearAE'
elif model_type_raw in {'ae', 'autoencoder', 'auto'}:
    model_type = 'AE'
else:
    model_type = 'AE' if hidden_dims is not None else 'linearAE'

if model_type == 'linearAE':
    hidden_dims = None
else:
    if hidden_dims is None:
        raise ValueError('hidden_dims missing for AE in results JSON')

input_dim = int(model_cfg.get('input_dim', N_FEATURES))
if input_dim != N_FEATURES:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={N_FEATURES}')

dropout_prob = model_cfg.get('dropout', 0.02)
if dropout_prob is None:
    dropout_prob = 0.0
dropout_prob = float(dropout_prob)

model = AutoEncoder(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dims=hidden_dims,
    dropout_prob=dropout_prob,
).to(device)

weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    if model_type == 'linearAE':
        weights_path = f'linear_AE_best_{run_name}.pt'
    else:
        weights_path = f'AE_best_{run_name}.pt'

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)
if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

# --- INIZIO AGGIUNTA: Caricamento statistiche di normalizzazione ---
stats_dir = weights_path.parent
train_mean_path = stats_dir / 'train_mean.pt'
train_std_path = stats_dir / 'train_std.pt'

if not train_mean_path.exists() or not train_std_path.exists():
    raise FileNotFoundError(f"File di normalizzazione mancanti in: {stats_dir}. Assicurati di averli copiati.")

# Carichiamo i tensori e assicuriamoci che siano sulla CPU
train_mean = torch.load(train_mean_path, map_location='cpu')
train_std = torch.load(train_std_path, map_location='cpu')

# Creiamo subito anche le copie NumPy per velocizzare la denormalizzazione
train_mean_np = train_mean.numpy()
train_std_np = train_std.numpy()
# --- FINE AGGIUNTA ---

print(f'Model type: {model_type} | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')
print(f'Loaded norm stats: {stats_dir.name}')

Model type: AE | latent_dim=288 | input_dim=65341
Loaded weights: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_288dim_norm0002\best_model.pt
Loaded norm stats: AE_288dim_norm0002


In [88]:
print(model)

AutoEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=65341, out_features=2048, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Dropout(p=0.0, inplace=False)
    (3): Linear(in_features=2048, out_features=1024, bias=True)
    (4): LeakyReLU(negative_slope=0.01)
    (5): Dropout(p=0.0, inplace=False)
    (6): Linear(in_features=1024, out_features=512, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Dropout(p=0.0, inplace=False)
    (9): Linear(in_features=512, out_features=288, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=288, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=2048, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=2048, out_features=65341, bias=True)
  )
)


## Step 5: Latent Space Analysis + Reconstruction Errors Analysis
Encode the matrices into the latent space and analyze feature distributions + Analyse MSE, MAE and Frobenius.

In [ ]:
# 1. Funzione di ricostruzione con gestione automatica del device
def compute_reconstruction(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 64, device=None):
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []
    reconstructions = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encoder(xb)
            latents.append(z.cpu().numpy())
            recon = model.decoder(z)
            reconstructions.append(recon.cpu().numpy())

    return np.concatenate(latents, axis=0), np.concatenate(reconstructions, axis=0)

# 2. Funzione per gli errori (rimane invariata, ora riceverà matrici quadrate corrette)
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    if original.shape != reconstructed.shape:
        raise ValueError(f'Input shapes mismatch: {original.shape} vs {reconstructed.shape}')

    diff = original - reconstructed

    
    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [90]:
# ==========================================
# ESECUZIONE DEL CALCOLO E RICOSTRUZIONE
# ==========================================


def process_dataset(dataset_name: str, matrix_file: Path):
    corr, indices, meta = load_corr_payload(matrix_file)
    all_np, x_all, n_assets, n_features, tril_idx = prepare_inputs(corr)

    if n_features != N_FEATURES:
        raise ValueError(
            f'Input dim mismatch for {dataset_name}: expected {N_FEATURES}, got {n_features}'
        )

    # --- MODIFICA 1: NORMALIZZAZIONE (Z-Score) ---
    # Normalizziamo le feature in ingresso usando le statistiche di training
    x_all_norm = (x_all - train_mean) / train_std

    # Passiamo x_all_norm al modello invece di x_all
    latents_all, recon_flat_norm = compute_reconstruction(model, x_all_norm, batch_size=64)

    # --- MODIFICA 2: DENORMALIZZAZIONE ---
    # Riportiamo le predizioni allo spazio originale moltiplicando per std e sommando la media
    recon_flat = (recon_flat_norm * train_std_np) + train_mean_np
    
    # Clip di sicurezza: garantisce che le correlazioni previste siano strettamente nel range [-1, 1]
    # prevenendo artefatti numerici dell'AE durante la valutazione dell'errore (MSE, MAE, ecc.)
    recon_flat = np.clip(recon_flat, -1.0, 1.0)
    # --------------------------------------

    n_matrices = all_np.shape[0]
    recon_all = np.zeros((n_matrices, n_assets, n_assets), dtype=np.float32)
    
    recon_all[:, tril_idx[0], tril_idx[1]] = recon_flat
    recon_all[:, tril_idx[1], tril_idx[0]] = recon_flat

    diag_idx = np.arange(n_assets)
    recon_all[:, diag_idx, diag_idx] = 1.0

    errors_df, summary_df = reconstruction_errors(all_np, recon_all)

    latent_dim = latents_all.shape[1]
    latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
    latent_df = pd.DataFrame(latents_all, columns=latent_cols)

    if indices is None:
        indices = np.arange(len(errors_df))

    if len(indices) != len(errors_df):
        raise ValueError(f'Error rows ({len(errors_df)}) do not match indices ({len(indices)})')
    errors_df.insert(0, 'matrix_idx', indices)

    print(f'\nReconstruction error summary ({dataset_name} data):')
    display(summary_df)

    errors_json_path = analysis_dir / f'reconstruction_errors_{dataset_name}_{RUN}.json'
    summary_json_path = analysis_dir / f'reconstruction_summary_{dataset_name}_{RUN}.json'
    errors_df.to_json(errors_json_path, orient='records', indent=2)
    summary_df.to_json(summary_json_path, orient='records', indent=2)

    print(f'Saved per-matrix errors: {errors_json_path}')
    print(f'Saved summary stats: {summary_json_path}')

    original_payload = torch.load(matrix_file, map_location='cpu')
    if isinstance(original_payload, dict):
        extended_payload = original_payload.copy()
    else:
        extended_payload = {'corr_tensor': original_payload}

    recon_tensor = torch.from_numpy(recon_all).float()
    extended_payload['corr_tensor_reconstructed'] = recon_tensor

    tickers = None
    if isinstance(meta, dict):
        tickers = meta.get('meta', {}).get('tickers', None)
    if tickers is not None:
        extended_payload['tickers'] = tickers

    output_path = analysis_dir / f'{dataset_name}_reconstructed_{RUN}.pt'
    torch.save(extended_payload, output_path)
    print(f'Saved reconstructed matrices: {output_path}')

    print(f'\nLatent distribution summary ({dataset_name} data):')
    display(latent_df.describe().T)

    valid_cols = [
        col for col in latent_cols
        if latent_df[col].notna().any() and latent_df[col].nunique() > 1
    ]
    latent_df = latent_df[valid_cols]

    if len(indices) != len(latent_df):
        raise ValueError(f'Latent rows ({len(latent_df)}) do not match indices ({len(indices)})')
    latent_df.insert(0, 'matrix_idx', indices)

    latent_json_path = analysis_dir / f'latent_{dataset_name}_{RUN}.json'
    latent_df.to_json(latent_json_path, orient='records', indent=2)
    print(f'Saved latent samples: {latent_json_path}')

    if len(valid_cols) < 2:
        print('Not enough valid latent dimensions for pairwise plots.')
    elif len(valid_cols) > 3:
        print(f'Skipping pairwise plot: too many latent dimensions ({len(valid_cols)} > 3).')
    else:
        plot_df = latent_df[valid_cols]
        grid = sns.PairGrid(plot_df, corner=True, diag_sharey=False)
        grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
        grid.figure.suptitle(
            f'Pairwise latent dimension plots ({dataset_name} data)',
            y=1.02,
        )
        pairplot_path = analysis_dir / f'latent_pairwise_{dataset_name}_{RUN}.png'
        grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved latent pairwise plot: {pairplot_path}')


for dataset_name, matrix_file in DATASET_FILES.items():
    print(f'\n=== Processing {dataset_name} ({matrix_file.name}) ===')
    process_dataset(dataset_name, matrix_file)


=== Processing train (train.pt) ===

Reconstruction error summary (train data):


,mean,std,min,median,max
MSE,0.000017,0.000010,5.400130e-07,0.000016,0.000060
MAE,0.002837,0.000799,5.519591e-04,0.002870,0.005234
Frobenius,1.427476,0.415208,2.660178e-01,1.435274,2.793953


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_288dim_norm0002\analysis_outputs\reconstruction_errors_train_AE_288dim_norm0002.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_288dim_norm0002\analysis_outputs\reconstruction_summary_train_AE_288dim_norm0002.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_288dim_norm0002\analysis_outputs\train_reconstructed_AE_288dim_norm0002.pt

Latent distribution summary (train data):


,count,mean,std,min,25%,50%,75%,max
z1,288.0,2.784544,7.853968,-13.430230,-3.857292,3.137631,8.560313,17.677296
z2,288.0,1.089596,10.943087,-18.295330,-10.921908,2.710656,10.290239,20.431709
z3,288.0,-8.630253,16.784811,-44.599464,-24.724156,-4.096397,3.132073,25.897224
z4,288.0,9.824886,12.163157,-20.600014,2.875410,9.328869,15.029469,39.684292
z5,288.0,8.056890,14.303434,-28.076145,-1.864295,6.190844,16.335179,41.180271
...,...,...,...,...,...,...,...,...
z284,288.0,-0.096304,7.137251,-25.156799,-4.313557,-0.010562,5.309062,15.271713
z285,288.0,2.875346,5.923352,-9.287532,-2.013483,1.615545,6.823849,17.501827
z286,288.0,-1.456868,8.097420,-23.820251,-4.186148,0.079716,4.719832,8.746004
z287,288.0,9.022749,13.985549,-10.642472,-2.137434,4.766014,17.138308,36.992947


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_288dim_norm0002\analysis_outputs\latent_train_AE_288dim_norm0002.json
Skipping pairwise plot: too many latent dimensions (288 > 3).
